# Brier Score Degradation Analysis

**Hypothesis**: Probes trained on the `[clues_end]` token maintain AUC when evaluated at later positions, but Brier score degrades. The degradation pattern differs between *simple* puzzles (no backtracking) and *hard* puzzles (with PUSH/POP tokens).

**Setup**: Candidate-set probes (9-class binary per cell) trained at step 0, evaluated across positions grouped by **number of empty cells** (`n_empty = 81 − n_filled`). This is a board-state–based axis rather than a sequence-position–based axis, making simple and hard puzzles directly comparable.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from sudoku.probes.session import ProbeSession
from sudoku.probes.modes import cell_candidates
from sudoku.probes.activations import build_grid_at_step
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN, END_CLUES_TOKEN, PAD_TOKEN_BT

In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
LAYER = 3     # layer to probe (0-indexed)
N_CELLS = 81    # probe all cells
SEED = 42

## Load session

In [ ]:
session = ProbeSession(CACHE_PATH)

## Classify puzzles: simple vs hard

Simple = no PUSH/POP tokens anywhere in the sequence (pure constraint propagation).  
Hard = at least one PUSH token (backtracking was used).

In [ ]:
is_simple = np.array([
    not any(t == PUSH_TOKEN for t in seq)
    for seq in session.sequences
])

n_simple = is_simple.sum()
n_hard = (~is_simple).sum()
print(f"Simple puzzles (no backtracking): {n_simple:,}")
print(f"Hard puzzles (with backtracking): {n_hard:,}")

# Sequence length distribution check
seq_lens = np.array([len(s) for s in session.sequences])
print(f"\nSimple seq len: min={seq_lens[is_simple].min()}, max={seq_lens[is_simple].max()}, mean={seq_lens[is_simple].mean():.1f}")
print(f"Hard   seq len: min={seq_lens[~is_simple].min()}, max={seq_lens[~is_simple].max()}, mean={seq_lens[~is_simple].mean():.1f}")

## Build train/test split at step 0

- **Train**: all puzzles at step 0 (80% split)
- **Test**: the remaining 20%, further divided into simple/hard subsets

In [ ]:
idx0 = session.index.at_step(1).first_per_puzzle()
train_mask, test_mask = session.split(idx0, test_size=0.2, seed=SEED)

train_idx = idx0[train_mask]
test_idx  = idx0[test_mask]

# Identify which test puzzles are simple vs hard
test_puzzle_ids = test_idx.puzzle_idx
test_is_simple = is_simple[test_puzzle_ids]

print(f"Train: {len(train_idx):,}  |  Test: {len(test_idx):,}")
print(f"Test simple: {test_is_simple.sum():,}  |  Test hard: {(~test_is_simple).sum():,}")

## Helper functions

In [ ]:
def build_candidate_targets(grids: list[str], cell_idx: int) -> tuple[np.ndarray, np.ndarray]:
    """Build candidate target vectors for a cell.
    
    Returns:
        targets: (N, 9) float — candidate bitvectors (only for empty cells)
        is_empty: (N,) bool — True where the cell is empty
    """
    is_empty = np.array([g[cell_idx] == "0" for g in grids])
    targets = np.zeros((len(grids), 9), dtype=np.float32)
    for i in np.where(is_empty)[0]:
        targets[i] = cell_candidates(grids[i], cell_idx)
    return targets, is_empty


def fit_cell_probe(X_train: np.ndarray, y_train: np.ndarray):
    """Fit a multi-label logistic regression for one cell (9 binary classifiers)."""
    clf = MultiOutputClassifier(LogisticRegression(C=1.0, max_iter=1000))
    clf.fit(X_train, y_train.astype(int))
    return clf


def eval_cell_probe(clf, X: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Evaluate probe: returns (mean_auc, mean_brier) across 9 digits."""
    if len(X) == 0:
        return float("nan"), float("nan")
    proba_list = clf.predict_proba(X)
    probas = np.column_stack([p[:, 1] for p in proba_list])  # (N, 9)
    aucs = []
    for d in range(9):
        if len(np.unique(y[:, d])) > 1:
            aucs.append(roc_auc_score(y[:, d], probas[:, d]))
    auc = float(np.mean(aucs)) if aucs else float("nan")
    brier = float(np.mean((probas - y) ** 2))
    return auc, brier

## Train probes at step 0

One multi-label LR probe per cell (81 probes total), trained on the training split at step 0.

In [ ]:
# Activations at step 0 for train and test splits
X_train_all = session.acts(train_idx, layer=LAYER)  # (n_train, d_model)
X_test_all  = session.acts(test_idx, layer=LAYER)   # (n_test,  d_model)

grids_train = session.grids(train_idx)
grids_test  = session.grids(test_idx)

print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

In [ ]:
# Train one probe per cell — filter to cells that have empty instances in train
cell_probes = {}  # cell_idx -> (clf, train_empty_mask)

for cell_idx in tqdm(range(N_CELLS), desc="Training probes"):
    targets_train, is_empty_train = build_candidate_targets(grids_train, cell_idx)
    if is_empty_train.sum() < 10:
        continue  # skip cells with almost no empty instances in train
    clf = fit_cell_probe(X_train_all[is_empty_train], targets_train[is_empty_train])
    cell_probes[cell_idx] = clf

print(f"Trained probes for {len(cell_probes)} cells")

## Evaluate by number of empty cells

Instead of step offset from `[clues_end]`, group samples by **how many cells are still empty** at the probed position (`n_empty = 81 − n_filled`).

- For simple puzzles: each fill step reduces n_empty by exactly 1, so this is a monotone reparameterisation.
- For hard (BT) puzzles: n_empty can fluctuate due to backtracking; using `first_per_puzzle()` selects the first moment the board reaches that fill count (forward pass, before any rollback).

Both groups are swept over the same n_empty axis, making the comparison fair.

In [ ]:
# Pre-identify test puzzle IDs for quick lookup
test_puzzle_id_set = set(test_puzzle_ids.tolist())
simple_test_puzzle_ids = set(test_puzzle_ids[test_is_simple].tolist())
hard_test_puzzle_ids   = set(test_puzzle_ids[~test_is_simple].tolist())

# n_filled in the index counts fill *tokens* seen up to a position, accounting
# for PUSH/POP stack resets.  For model-generated BT sequences a re-fill after
# rollback can push the counter above 81.  Cap the sweep at [0, 81].
all_n_empty = 81 - session.index.n_filled

# Starting n_empty: largest value among step-0 samples (= 81 - n_clues)
n_empty_max = int(all_n_empty[session.index.step == 0].max())
n_empty_values = list(range(n_empty_max, -1, -1))  # high → low, stopping at 0
print(f"Sweeping n_empty from {n_empty_max} down to 0 ({len(n_empty_values)} points)")

results = {"simple": {"auc": [], "brier": [], "n": [], "n_empty": []},
           "hard":   {"auc": [], "brier": [], "n": [], "n_empty": []}}

for n_empty in tqdm(n_empty_values, desc="Evaluating n_empty"):
    n_filled_target = 81 - n_empty
    # where_filled matches the stack-adjusted counter; first_per_puzzle picks the
    # *first* forward visit (before any rollback) for BT sequences.
    idx_slot = session.index.where_filled(n_filled_target).first_per_puzzle()
    # Filter to test puzzle IDs
    in_test = np.isin(idx_slot.puzzle_idx, list(test_puzzle_id_set))
    idx_slot = idx_slot[in_test]

    if len(idx_slot) == 0:
        for grp in results:
            results[grp]["auc"].append(float("nan"))
            results[grp]["brier"].append(float("nan"))
            results[grp]["n"].append(0)
            results[grp]["n_empty"].append(n_empty)
        continue

    slot_is_simple = np.isin(idx_slot.puzzle_idx, list(simple_test_puzzle_ids))
    masks = {"simple": slot_is_simple, "hard": ~slot_is_simple}

    X_slot = session.acts(idx_slot, layer=LAYER)
    grids_slot = session.grids(idx_slot)

    for grp, mask in masks.items():
        results[grp]["n_empty"].append(n_empty)
        if mask.sum() == 0:
            results[grp]["auc"].append(float("nan"))
            results[grp]["brier"].append(float("nan"))
            results[grp]["n"].append(0)
            continue

        X_grp = X_slot[mask]
        grids_grp = [grids_slot[i] for i in np.where(mask)[0]]

        cell_aucs, cell_briers = [], []
        for cell_idx, clf in cell_probes.items():
            targets, is_empty_mask = build_candidate_targets(grids_grp, cell_idx)
            if is_empty_mask.sum() < 5:
                continue
            auc, brier = eval_cell_probe(clf, X_grp[is_empty_mask], targets[is_empty_mask])
            if not np.isnan(auc):
                cell_aucs.append(auc)
            if not np.isnan(brier):
                cell_briers.append(brier)

        results[grp]["auc"].append(float(np.mean(cell_aucs)) if cell_aucs else float("nan"))
        results[grp]["brier"].append(float(np.mean(cell_briers)) if cell_briers else float("nan"))
        results[grp]["n"].append(int(mask.sum()))

print("Done.")

## Plot results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Candidate-set probe (layer {LAYER}) trained at step 0 — evaluated by n_empty", fontsize=13)

colors = {"simple": "steelblue", "hard": "tomato"}
labels = {"simple": "Simple (no backtracking)", "hard": "Hard (backtracking)"}

for grp in ["simple", "hard"]:
    xs     = results[grp]["n_empty"]
    aucs   = results[grp]["auc"]
    briers = results[grp]["brier"]
    ns     = results[grp]["n"]

    valid_a = [(x, a) for x, a in zip(xs, aucs) if not np.isnan(a) and results[grp]["n"][xs.index(x)] > 0]
    valid_b = [(x, b) for x, b in zip(xs, briers) if not np.isnan(b) and results[grp]["n"][xs.index(x)] > 0]

    xa, ya = zip(*valid_a) if valid_a else ([], [])
    xb, yb = zip(*valid_b) if valid_b else ([], [])

    mean_n = int(np.mean([n for n in ns if n > 0])) if any(n > 0 for n in ns) else 0
    axes[0].plot(xa, ya, color=colors[grp], linewidth=1.8, marker="o", markersize=3,
                 label=f"{labels[grp]} (n≈{mean_n})")
    axes[1].plot(xb, yb, color=colors[grp], linewidth=1.8, marker="o", markersize=3)

for ax in axes:
    ax.set_xlabel("Number of empty cells (n_empty = 81 − n_filled)")
    ax.invert_xaxis()   # left = many empty (start of solve), right = few empty (end)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Mean AUC (across cells and digits)")
axes[0].set_title("AUC vs n_empty")
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)

axes[1].set_ylabel("Mean Brier score (lower = better)")
axes[1].set_title("Brier Score vs n_empty")
axes[1].legend(
    handles=[plt.Line2D([0], [0], color=colors[grp], linewidth=2) for grp in ["simple", "hard"]],
    labels=[labels[grp] for grp in ["simple", "hard"]],
)

plt.tight_layout()
plt.savefig("brier_degradation_by_nempty.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved brier_degradation_by_nempty.png")

## Numerical summary

In [ ]:
print(f"{'n_empty':>7}  {'AUC_simple':>10}  {'AUC_hard':>8}  {'Brier_simple':>12}  {'Brier_hard':>10}  {'N_simple':>8}  {'N_hard':>6}")
print("-" * 75)
for i, n_e in enumerate(results["simple"]["n_empty"]):
    def _v(grp, key): return results[grp][key][i]
    print(f"{n_e:>7}  {_v('simple','auc'):>10.4f}  {_v('hard','auc'):>8.4f}  "
          f"{_v('simple','brier'):>12.4f}  {_v('hard','brier'):>10.4f}  "
          f"{_v('simple','n'):>8}  {_v('hard','n'):>6}")

In [ ]:
def get_puzzle_at_step(session, puzzle_id: int, step: int):
    idx = session.index.at_step(step)

    idx = idx[idx.puzzle_idx == puzzle_id]
    if len(idx) == 0:
        raise ValueError(f"Puzzle {puzzle_id} never reaches step {step} "
                         f"(seq len={len(session.sequences[puzzle_id])})")
    return idx[:1]

def get_puzzle_at_nempty(session, puzzle_id: int, n_empty: int):
    """Return a single-sample ActivationIndex for puzzle_id at n_empty empty cells.

    Selects the *first* position in the trace where exactly n_empty cells remain
    unfilled (forward pass; ignores later revisits due to backtracking).

    Raises ValueError if the puzzle never reaches that fill count.
    """
    idx = session.index.where_filled(81 - n_empty)
    idx = idx[idx.puzzle_idx == puzzle_id]
    if len(idx) == 0:
        raise ValueError(
            f"Puzzle {puzzle_id} never reaches n_empty={n_empty} "
            f"(seq len={len(session.sequences[puzzle_id])})"
        )
    return idx[:1]   # first occurrence = forward pass


def show_probe_state(session, cell_probes,
                     idx_entry=None,
                     puzzle_id: int | None = None,
                     n_empty: int | None = None,
                     layer=LAYER):
    """Visualise probe candidate predictions for one board snapshot.

    Supply either:
      idx_entry                — a single-sample ActivationIndex (use slice [:1])
      puzzle_id  +  n_empty   — convenience; calls get_puzzle_at_nempty internally

    Each empty cell shows a 3×3 sub-grid of per-digit predicted probabilities
    (green=high, red=low).  Bold digit = true candidate per current board state.
    Filled cells are shown in grey with their digit centred.
    """
    if idx_entry is None:
        if puzzle_id is None or n_empty is None:
            raise ValueError("Supply either idx_entry or both puzzle_id and n_empty")
        idx_entry = get_puzzle_at_step(session, puzzle_id, n_empty)

    assert len(idx_entry) == 1, "Pass a single-sample index (use idx[:1] not idx[0])"

    X        = session.acts(idx_entry, layer=layer)  # (1, d_model)
    grid_str = session.grids(idx_entry)[0]            # 81-char board-state string

    fig, ax = plt.subplots(figsize=(9.5, 9.5))
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 9)
    ax.set_aspect('equal')
    ax.axis('off')

    cmap = plt.cm.RdYlGn

    for cell_idx in range(81):
        r, c = divmod(cell_idx, 9)
        y0   = 8 - r   # bottom-left y of this cell in plot coords
        ch   = grid_str[cell_idx]

        if ch != '0':
            ax.add_patch(plt.Rectangle((c, y0), 1, 1, color='#cccccc', zorder=1))
            ax.text(c + 0.5, y0 + 0.5, ch,
                    ha='center', va='center', fontsize=22, fontweight='bold', zorder=2)

        elif cell_idx in cell_probes:
            clf        = cell_probes[cell_idx]
            proba_list = clf.predict_proba(X)
            probas     = np.array([p[0, 1] for p in proba_list])  # (9,)
            true_cands = cell_candidates(grid_str, cell_idx)       # list[int] length 9

            for d in range(9):
                dr, dc = divmod(d, 3)
                sx = c + dc / 3
                sy = y0 + (2 - dr) / 3   # dr=0 → top row → largest y
                ax.add_patch(plt.Rectangle((sx, sy), 1/3, 1/3,
                                           color=cmap(float(probas[d])), zorder=1))
                ax.text(sx + 1/6, sy + 1/6, str(d + 1),
                        ha='center', va='center', fontsize=7, zorder=2,
                        fontweight='bold' if true_cands[d] else 'normal',
                        color='black'     if true_cands[d] else '#666666')
        else:
            ax.text(c + 0.5, y0 + 0.5, '?',
                    ha='center', va='center', fontsize=14, color='grey')

    for i in range(10):
        lw = 2.5 if i % 3 == 0 else 0.5
        ax.axhline(i, color='black', linewidth=lw, zorder=3)
        ax.axvline(i, color='black', linewidth=lw, zorder=3)

    n_empty_actual = grid_str.count('0')
    ax.set_title(
        f"Puzzle {int(idx_entry.puzzle_idx[0])}  |  "
        f"seq_pos={int(idx_entry.seq_pos[0])}  |  n_empty={n_empty_actual} | token={idx_entry.tokens[0]}\n"
        "Background: P(candidate) — green=1, red=0  |  Bold digit = true candidate",
        fontsize=11, pad=12,
    )
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.02, label='P(candidate)')
    plt.tight_layout()
    plt.savefig('backtrack.svg')
    plt.show()


# --- Example: track a specific simple test puzzle across n_empty values ---
_example_id = 16
print(f"Tracking simple test puzzle id={_example_id}")
for _ne in [10]:
    show_probe_state(session, cell_probes, puzzle_id=_example_id, n_empty=_ne)

In [ ]:
565 561

In [ ]:
import ipywidgets as widgets

@widgets.interact(
    puzzle_id=widgets.Dropdown(
        options=sorted(hard_test_puzzle_ids),
        description="Puzzle:",
    ),
    n_empty=widgets.IntSlider(
        value=40, min=0, max=100, step=1,
        description="step:",
        continuous_update=False,   # only redraw on release, not while dragging
    ),
)
def _show_interactive(puzzle_id, n_empty):
    try:
        show_probe_state(session, cell_probes, puzzle_id=puzzle_id, n_empty=n_empty)
    except ValueError as e:
        print(e)

In [ ]:
def get_probe_weight(cell_probes, cell_idx: int, digit: int) -> np.ndarray:
    """Return the weight vector (d_model,) for the binary LR probe at (cell, digit).
    digit is 1-indexed (1–9).
    """
    return cell_probes[cell_idx].estimators_[digit - 1].coef_[0]


def plot_probe_cosine_similarity(cell_probes, cell_idx: int, digit: int):
    """Plot cosine similarity of the probe w-vector for (cell_idx, digit)
    against every other cell's probe w-vector for the same digit.

    The reference cell is outlined in yellow.  Cells without a trained probe
    are left white (NaN).
    """
    if cell_idx not in cell_probes:
        raise ValueError(f"No probe trained for cell {cell_idx}")

    w_ref  = get_probe_weight(cell_probes, cell_idx, digit)
    norm_ref = np.linalg.norm(w_ref)

    sim_grid = np.full(81, np.nan)
    for c, _ in cell_probes.items():
        w = get_probe_weight(cell_probes, c, digit)
        sim_grid[c] = np.dot(w_ref, w) / (norm_ref * np.linalg.norm(w))

    sim_matrix = sim_grid.reshape(9, 9)

    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)

    for r in range(9):
        for c in range(9):
            v = sim_matrix[r, c]
            if not np.isnan(v):
                ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(v) > 0.65 else "black")

    # Highlight reference cell
    ref_r, ref_c = divmod(cell_idx, 9)
    ax.add_patch(plt.Rectangle(
        (ref_c - 0.5, ref_r - 0.5), 1, 1,
        fill=False, edgecolor="yellow", linewidth=3, zorder=3,
    ))

    # 3×3 block boundaries
    for pos in [2.5, 5.5]:
        ax.axhline(pos, color="black", linewidth=2)
        ax.axvline(pos, color="black", linewidth=2)

    ax.set_xticks(range(9))
    ax.set_yticks(range(9))
    ax.set_title(
        f"Probe cosine similarity — digit {digit},  "
        f"reference cell {cell_idx} (r{ref_r} c{ref_c})",
        fontsize=11,
    )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="cosine similarity")
    plt.tight_layout()
    plt.savefig("probe_cosine_sim.svg")
    plt.show()


# --- Interactive widget ---
@widgets.interact(
    cell_idx=widgets.IntSlider(value=0, min=0, max=80, step=1,
                               description="Cell:", continuous_update=False),
    digit=widgets.IntSlider(value=7, min=1, max=9, step=1,
                            description="Digit:", continuous_update=False),
)
def _show_cosine(cell_idx, digit):
    if cell_idx not in cell_probes:
        print(f"No probe for cell {cell_idx}")
        return
    plot_probe_cosine_similarity(cell_probes, cell_idx, digit)

## Candidates per substructure

Train one multi-label LR probe per substructure (9 rows × 9 cols × 9 boxes = 27 probes) on the training split at step 0. Each probe predicts which digits are **candidates** (i.e. *not yet present*) in that substructure. A fully-filled substructure has no candidates. Evaluate average Brier score and AUC as a function of n_empty, split by simple vs hard puzzles.

In [ ]:

def eval_structure_probe(clf: MultiOutputClassifier, X: 'np.ndarray', y: 'np.ndarray') -> 'tuple[float, float]':
    """Returns (mean_auc, mean_brier) across 9 digits for a structure probe."""
    if len(X) == 0:
        return float('nan'), float('nan')
    proba_list = clf.predict_proba(X)
        
    probas = np.column_stack([p[:, 1] for p in proba_list])  # (N, 9)
    aucs = []
    for d in range(9):
        if len(np.unique(y[:, d])) > 1:
            aucs.append(roc_auc_score(y[:, d], probas[:, d]))
    auc = float(np.mean(aucs)) if aucs else float('nan')
    brier = float(np.mean((probas - y) ** 2))

    return auc, brier



In [ ]:
# --- Helper functions ---

def build_structure_targets(grids: list, subtype: str, sidx: int) -> 'np.ndarray':
    """(N, 9) binary — targets[i, d] = 1 if digit d+1 is a candidate (not present) in the substructure."""
    n = len(grids)
    targets = np.ones((n, 9), dtype=np.float32)  # start: all digits are candidates
    for i, g in enumerate(grids):
        if subtype == 'row':
            cells = g[sidx * 9:(sidx + 1) * 9]
        elif subtype == 'col':
            cells = [g[r * 9 + sidx] for r in range(9)]
        else:  # box
            br, bc = (sidx // 3) * 3, (sidx % 3) * 3
            cells = [g[(br + dr) * 9 + (bc + dc)] for dr in range(3) for dc in range(3)]
        for ch in cells:
            if ch in '123456789':
                targets[i, int(ch) - 1] = 0.0  # digit placed → no longer a candidate
    return targets

# --- Train one probe per substructure at step 0 ---
SUBTYPES = (
    [('row', i) for i in range(9)]
    + [('col', i) for i in range(9)]
    + [('box', i) for i in range(9)]
)

struct_probes = {}  # (subtype, sidx) -> clf
for subtype, sidx in tqdm(SUBTYPES, desc='Training structure probes'):
    y_tr = build_structure_targets(grids_train, subtype, sidx)
    clf = MultiOutputClassifier(LogisticRegression(C=1.0, max_iter=1000))
    clf.fit(X_train_all, y_tr.astype(int))
    struct_probes[(subtype, sidx)] = clf

print(f'Trained {len(struct_probes)} structure probes')


In [ ]:
struct_results = {
    'simple': {'auc': [], 'brier': [], 'n': [], 'n_empty': []},
    'hard':   {'auc': [], 'brier': [], 'n': [], 'n_empty': []},
}

for n_empty in tqdm(n_empty_values, desc='Evaluating structure probes'):
    n_filled_target = 81 - n_empty
    idx_slot = session.index.where_filled(n_filled_target).first_per_puzzle()
    in_test = np.isin(idx_slot.puzzle_idx, list(test_puzzle_id_set))
    idx_slot = idx_slot[in_test]

    if len(idx_slot) == 0:
        for grp in struct_results:
            struct_results[grp]['auc'].append(float('nan'))
            struct_results[grp]['brier'].append(float('nan'))
            struct_results[grp]['n'].append(0)
            struct_results[grp]['n_empty'].append(n_empty)
        continue

    slot_is_simple = np.isin(idx_slot.puzzle_idx, list(simple_test_puzzle_ids))
    masks = {'simple': slot_is_simple, 'hard': ~slot_is_simple}

    X_slot = session.acts(idx_slot, layer=LAYER)
    grids_slot = session.grids(idx_slot)

    for grp, mask in masks.items():
        struct_results[grp]['n_empty'].append(n_empty)
        if mask.sum() == 0:
            struct_results[grp]['auc'].append(float('nan'))
            struct_results[grp]['brier'].append(float('nan'))
            struct_results[grp]['n'].append(0)
            continue

        X_grp = X_slot[mask]
        grids_grp = [grids_slot[i] for i in np.where(mask)[0]]

        sub_aucs, sub_briers = [], []
        for (subtype, sidx), clf in struct_probes.items():
            y = build_structure_targets(grids_grp, subtype, sidx)
            auc, brier = eval_structure_probe(clf, X_grp, y)
            if not np.isnan(auc):
                sub_aucs.append(auc)
            if not np.isnan(brier):
                sub_briers.append(brier)

        struct_results[grp]['auc'].append(float(np.mean(sub_aucs)) if sub_aucs else float('nan'))
        struct_results[grp]['brier'].append(float(np.mean(sub_briers)) if sub_briers else float('nan'))
        struct_results[grp]['n'].append(int(mask.sum()))

print('Done.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f'Structure probe (layer {LAYER}) trained at step 0 — evaluated by n_empty',
    fontsize=13,
)

colors = {'simple': 'steelblue', 'hard': 'tomato'}
labels = {'simple': 'Simple (no backtracking)', 'hard': 'Hard (backtracking)'}

for grp in ['simple', 'hard']:
    xs     = struct_results[grp]['n_empty']
    aucs   = struct_results[grp]['auc']
    briers = struct_results[grp]['brier']
    ns     = struct_results[grp]['n']

    valid_a = [(x, a) for x, a, n in zip(xs, aucs, ns) if not np.isnan(a) and n > 0]
    valid_b = [(x, b) for x, b, n in zip(xs, briers, ns) if not np.isnan(b) and n > 0]

    xa, ya = zip(*valid_a) if valid_a else ([], [])
    xb, yb = zip(*valid_b) if valid_b else ([], [])

    mean_n = int(np.mean([n for n in ns if n > 0])) if any(n > 0 for n in ns) else 0
    axes[0].plot(xa, ya, color=colors[grp], linewidth=1.8, marker='o', markersize=3,
                 label=f'{labels[grp]} (n≈{mean_n})')
    axes[1].plot(xb, yb, color=colors[grp], linewidth=1.8, marker='o', markersize=3)

for ax in axes:
    ax.set_xlabel('Number of empty cells (n_empty = 81 − n_filled)')
    ax.invert_xaxis()
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Mean AUC (across 27 substructures and 9 digits)')
axes[0].set_title('AUC vs n_empty')
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)

axes[1].set_ylabel('Mean Brier score (lower = better)')
axes[1].set_title('Brier Score vs n_empty')
axes[1].legend(
    handles=[plt.Line2D([0], [0], color=colors[grp], linewidth=2) for grp in ['simple', 'hard']],
    labels=[labels[grp] for grp in ['simple', 'hard']],
)

plt.tight_layout()
plt.savefig('brier_degradation_structure_by_nempty.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved brier_degradation_structure_by_nempty.png')


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

from sudoku.data_bt import (
    END_CLUES_TOKEN, PAD_TOKEN_BT, PUSH_TOKEN, POP_TOKEN, SUCCESS_TOKEN
)
from sudoku.probes.activations import build_grid_at_step


def _decode_token(tok):
    """Return human-readable string for a token."""
    if 0 <= tok <= 728:
        digit = (tok % 9) + 1
        col   = (tok // 9) % 9
        row   = tok // 81
        return f"(col={col}, row={row}, digit={digit})"
    elif tok == END_CLUES_TOKEN: return "<end_clues>"
    elif tok == PAD_TOKEN_BT:    return "<pad>"
    elif tok == PUSH_TOKEN:      return "<push>"
    elif tok == POP_TOKEN:       return "<pop>"
    elif tok == SUCCESS_TOKEN:   return "<success>"
    else:                        return f"<unknown:{tok}>"


def _get_seq_len(session, puzzle_id):
    seq = session.sequences[puzzle_id]
    for i, tok in enumerate(seq):
        if tok == PAD_TOKEN_BT:
            return i
    return len(seq)


def _struct_true_candidates(grid_str, subtype, sidx):
    """Boolean (9,): True if digit d+1 is still a candidate in this substructure."""
    candidates = np.ones(9, dtype=bool)
    if subtype == "row":
        cells = grid_str[sidx * 9:(sidx + 1) * 9]
    elif subtype == "col":
        cells = [grid_str[r * 9 + sidx] for r in range(9)]
    else:  # box
        br, bc = (sidx // 3) * 3, (sidx % 3) * 3
        cells = [grid_str[(br + dr) * 9 + (bc + dc)] for dr in range(3) for dc in range(3)]
    for ch in cells:
        if ch in "123456789":
            candidates[int(ch) - 1] = False
    return candidates


def _struct_proba(struct_probes, subtype, sidx, X):
    clf = struct_probes.get((subtype, sidx))
    if clf is None:
        return np.full(9, float("nan"))
    
    # MODIFIED
    proba_list = clf.predict_proba(X)
    
    
    
    return np.array([p[0, 1] for p in proba_list])


def _cell(ax, x0, y0, prob, is_cand, label, cmap):
    ax.add_patch(plt.Rectangle((x0, y0), 1, 1, color=cmap(prob), zorder=1))
    ax.text(x0 + 0.5, y0 + 0.63, label,
            ha="center", va="center", fontsize=7, zorder=2,
            fontweight="bold" if is_cand else "normal",
            color="#111" if is_cand else "#888")
    ax.text(x0 + 0.5, y0 + 0.24, f"{prob:.2f}",
            ha="center", va="center", fontsize=5, zorder=2, color="#111")


def _grid_lines(ax, w, h, thick_every=3):
    for i in range(h + 1):
        lw = 1.8 if i % thick_every == 0 else 0.4
        ax.plot([0, w], [i, i], color="black", linewidth=lw, zorder=3)
    for i in range(w + 1):
        lw = 1.8 if i % thick_every == 0 else 0.4
        ax.plot([i, i], [0, h], color="black", linewidth=lw, zorder=3)


def _draw_board(ax, grid_str):
    ax.set_xlim(0, 9); ax.set_ylim(0, 9)
    ax.set_aspect("equal"); ax.axis("off")
    for cell_idx in range(81):
        r, c = divmod(cell_idx, 9)
        y0   = 8 - r
        ch   = grid_str[cell_idx]
        ax.add_patch(plt.Rectangle((c, y0), 1, 1,
                                   color="#d8d8d8" if ch != "0" else "white", zorder=1))
        if ch != "0":
            ax.text(c + 0.5, y0 + 0.5, ch, ha="center", va="center",
                    fontsize=16, fontweight="bold", zorder=2)
    _grid_lines(ax, 9, 9)


def _draw_row_grid(ax, grid_str, struct_probes, X):
    """9 rows × 9 digit-columns. Row index on left, digit header on top."""
    cmap = plt.cm.RdYlGn
    ax.set_xlim(-0.55, 9.05); ax.set_ylim(-0.1, 9.55)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Row candidates + probe score", fontsize=8, pad=3)

    for d in range(9):
        ax.text(d + 0.5, 9.35, str(d + 1), ha="center", va="center", fontsize=6.5, color="#444")

    for sidx in range(9):
        true_cands = _struct_true_candidates(grid_str, "row", sidx)
        probas     = _struct_proba(struct_probes, "row", sidx, X)
        y0 = 8 - sidx
        ax.text(-0.2, y0 + 0.5, f"R{sidx}", ha="right", va="center", fontsize=6.5, color="#444")
        for d in range(9):
            _cell(ax, d, y0, float(probas[d]), bool(true_cands[d]), str(d + 1), cmap)

    _grid_lines(ax, 9, 9)


def _draw_col_grid(ax, grid_str, struct_probes, X):
    """9 column-strips across × 9 digit-rows down. Column header on top, digit on left."""
    cmap = plt.cm.RdYlGn
    ax.set_xlim(-0.1, 9.55); ax.set_ylim(-0.55, 9.05)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Column candidates + probe score", fontsize=8, pad=15)

    for sidx in range(9):
        ax.text(sidx + 0.5, 9.35, f"C{sidx}", ha="center", va="center", fontsize=6.5, color="#444")

    for d in range(9):
        ax.text(-0.2, 8 - d + 0.5, str(d + 1), ha="right", va="center", fontsize=6.5, color="#444")

    for sidx in range(9):          # x = column index
        true_cands = _struct_true_candidates(grid_str, "col", sidx)
        probas     = _struct_proba(struct_probes, "col", sidx, X)
        for d in range(9):         # y = digit (8-d so digit 1 is at top)
            _cell(ax, sidx, 8 - d, float(probas[d]), bool(true_cands[d]), str(d + 1), cmap)

    _grid_lines(ax, 9, 9)


def _draw_box_grid(ax, grid_str, struct_probes, X):
    """3×3 arrangement of boxes, each box contains a 3×3 digit sub-grid (digits 1–9)."""
    cmap = plt.cm.RdYlGn
    ax.set_xlim(-0.1, 9.55); ax.set_ylim(-0.1, 9.55)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Box candidates + probe score", fontsize=8, pad=3)

    for sidx in range(9):          # box index 0..8, row-major (top-left first)
        br, bc = sidx // 3, sidx % 3
        true_cands = _struct_true_candidates(grid_str, "box", sidx)
        probas     = _struct_proba(struct_probes, "box", sidx, X)

        # Box origin in plot coords (y increases upward; br=0 → top → largest y)
        box_x0 = bc * 3
        box_y0 = (2 - br) * 3

        # Box label centred above the box
        ax.text(box_x0 + 1.5, box_y0 + 3.2, f"B{sidx}",
                ha="center", va="center", fontsize=6.5, color="#444")

        for d in range(9):         # digits 1–9 arranged 3×3 within the box
            dc = d % 3             # digit column within box
            dr = d // 3            # digit row within box (0 = top)
            x0 = box_x0 + dc
            y0 = box_y0 + (2 - dr)
            _cell(ax, x0, y0, float(probas[d]), bool(true_cands[d]), str(d + 1), cmap)

    # Fine grid inside boxes (thin), thick box boundaries
    for i in range(10):
        lw = 2.0 if i % 3 == 0 else 0.4
        ax.plot([0, 9], [i, i], color="black", linewidth=lw, zorder=3)
        ax.plot([i, i], [0, 9], color="black", linewidth=lw, zorder=3)


def _show_puzzle_step(puzzle_id, seq_pos):
    seq        = session.sequences[puzzle_id]
    actual_len = _get_seq_len(session, puzzle_id)
    if seq_pos >= actual_len:
        print(f"seq_pos {seq_pos} >= sequence length {actual_len}")
        return

    tok        = int(seq[seq_pos])
    grid_str   = build_grid_at_step(seq, seq_pos)
    X          = session.activations[puzzle_id, LAYER, seq_pos][np.newaxis, :]

    fig, axes = plt.subplots(2, 2, figsize=(12, 9.5),
                              gridspec_kw={"hspace": 0.22, "wspace": 0.18})
    fig.suptitle(
        f"Puzzle {puzzle_id}  │  seq_pos={seq_pos}  │  "
        f"token={tok}  │  {_decode_token(tok)}",
        fontsize=10,
    )

    _draw_board(axes[0, 0], grid_str)
    axes[0, 0].set_title(f"Board state  (filled={81 - grid_str.count(chr(48))})",
                         fontsize=8, pad=3)

    _draw_row_grid(axes[0, 1], grid_str, struct_probes, X)
    _draw_col_grid(axes[1, 0], grid_str, struct_probes, X)
    _draw_box_grid(axes[1, 1], grid_str, struct_probes, X)

    extent = axes[0,1].get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    fig.savefig('only_ax1.svg', bbox_inches=extent, dpi=300)
    extent = axes[1,0].get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    fig.savefig('only_ax2.svg', bbox_inches=extent, dpi=300)

    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=plt.Normalize(0, 1))
    sm.set_array([])
    fig.colorbar(sm, ax=[axes[0, 1], axes[1, 0], axes[1, 1]],
                 fraction=0.03, pad=0.02, label="P(candidate)")
    plt.show()


# ---- Widgets ----
_all_puzzle_ids = sorted(set(int(p) for p in session.index.puzzle_idx))

_puzzle_dd = widgets.Dropdown(
    options=_all_puzzle_ids,
    value=_all_puzzle_ids[0],
    description="Puzzle ID:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)

_seq_pos_slider = widgets.IntSlider(
    value=0, min=0, max=_get_seq_len(session, _all_puzzle_ids[0]) - 1, step=1,
    description="Seq pos:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

def _update_slider_range(change):
    pid  = change["new"]
    slen = _get_seq_len(session, pid)
    _seq_pos_slider.max = slen - 1
    if _seq_pos_slider.value >= slen:
        _seq_pos_slider.value = 0

_puzzle_dd.observe(_update_slider_range, names="value")

_out = widgets.interactive_output(
    _show_puzzle_step,
    {"puzzle_id": _puzzle_dd, "seq_pos": _seq_pos_slider},
)

display(widgets.VBox([_puzzle_dd, _seq_pos_slider, _out]))


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

from sudoku.activations import load_checkpoint
from sudoku.data_bt import END_CLUES_TOKEN, PAD_TOKEN_BT, PUSH_TOKEN, POP_TOKEN, SUCCESS_TOKEN
from sudoku.probes.activations import build_grid_at_step
from sudoku.probes.modes import cell_candidates

# ─── 1. Load model checkpoint ────────────────────────────────────────────────
# Derive checkpoint dir from the CACHE_PATH already in scope.
_ckpt_dir = CACHE_PATH.replace("activations.npz", "checkpoint")
_params, _model = load_checkpoint(_ckpt_dir)

# ─── 2. Extract final LN + lm_head weights (float32 numpy) ──────────────────
# Activations are stored as last-block residual stream (before the final
# LayerNorm and lm_head).  Applying those two ops recovers the logits without
# running the full forward pass again.
_ln_scale  = np.array(_params["LayerNorm_0"]["scale"], dtype=np.float32)
_ln_bias   = np.array(_params["LayerNorm_0"]["bias"],  dtype=np.float32)
_lm_kernel = np.array(_params["lm_head"]["kernel"],    dtype=np.float32)  # (d_model, vocab)
_lm_bias   = np.array(_params["lm_head"]["bias"],      dtype=np.float32)  # (vocab,)
_LAST_LAYER = session.activations.shape[1] - 1  # index of last transformer block

def _logits(puzzle_id: int, seq_pos: int) -> np.ndarray:
    """Final LN + lm_head applied to stored last-layer activations → (vocab_size,)."""
    x = np.array(session.activations[puzzle_id, _LAST_LAYER, seq_pos], dtype=np.float32)
    x = (x - x.mean()) / np.sqrt(x.var() + 1e-6) * _ln_scale + _ln_bias
    return x @ _lm_kernel + _lm_bias

# ─── 3. Token utilities ───────────────────────────────────────────────────────
_SPECIAL = {
    END_CLUES_TOKEN: "<end_clues>",
    PAD_TOKEN_BT:    "<pad>",
    PUSH_TOKEN:      "<push>",
    POP_TOKEN:       "<pop>",
    SUCCESS_TOKEN:   "<success>",
}

def _decode(tok: int) -> str:
    if 0 <= tok <= 728:
        return f"fill(r={(tok//81)},c={(tok//9)%9},d={(tok%9)+1})"
    return _SPECIAL.get(tok, f"<unk:{tok}>")

def _seq_len(puzzle_id: int) -> int:
    for i, t in enumerate(session.sequences[puzzle_id]):
        if int(t) == PAD_TOKEN_BT:
            return i
    return len(session.sequences[puzzle_id])

def _token_options(puzzle_id: int):
    seq = session.sequences[puzzle_id]
    n   = _seq_len(puzzle_id)
    return [(f"{i:3d}:  {_decode(int(seq[i]))}", i) for i in range(n)]

# ─── 4. Drawing ───────────────────────────────────────────────────────────────
def _draw_board(ax, grid_str: str) -> None:
    """9×9 board: filled cells shaded + digit; empty cells show valid candidates."""
    ax.set_xlim(0, 9); ax.set_ylim(0, 9)
    ax.set_aspect("equal"); ax.axis("off")

    for cell_idx in range(81):
        r, c   = divmod(cell_idx, 9)
        y0     = 8 - r
        ch     = grid_str[cell_idx]
        if ch != "0":
            ax.add_patch(plt.Rectangle((c, y0), 1, 1, color="#d8d8d8", zorder=1))
            ax.text(c + 0.5, y0 + 0.5, ch, ha="center", va="center",
                    fontsize=18, fontweight="bold", zorder=2)
        else:
            ax.add_patch(plt.Rectangle((c, y0), 1, 1, color="white", zorder=1))
            cands = cell_candidates(grid_str, cell_idx)   # (9,) binary list
            for d in range(9):
                if cands[d]:
                    dc, dr = d % 3, d // 3
                    ax.text(c + dc/3 + 1/6, y0 + (2 - dr)/3 + 1/6,
                            str(d + 1), ha="center", va="center",
                            fontsize=5.5, color="#222", fontweight="bold", zorder=2)

    for i in range(10):
        lw = 2.2 if i % 3 == 0 else 0.5
        ax.axhline(i, color="black", linewidth=lw, zorder=3)
        ax.axvline(i, color="black", linewidth=lw, zorder=3)


def _draw_top10(ax, logits_vec: np.ndarray) -> None:
    """Horizontal bar chart of top-10 softmax probabilities."""
    shifted   = logits_vec - logits_vec.max()
    probs     = np.exp(shifted) / np.exp(shifted).sum()
    top_i     = np.argsort(probs)[-10:][::-1]   # best first
    top_p     = probs[top_i]
    labels    = [_decode(int(t)) for t in top_i]
    colors    = ["#2196F3" if int(t) <= 728 else "#FF9800" for t in top_i]

    # Plot with rank 0 at the top → reverse for barh
    ax.barh(range(10), top_p[::-1], color=colors[::-1])
    ax.set_yticks(range(10))
    ax.set_yticklabels(labels[::-1], fontsize=8, fontfamily="monospace")
    ax.set_xlabel("Softmax probability", fontsize=8)
    ax.set_title("Top-10 model predictions", fontsize=9, pad=4)
    ax.tick_params(axis="x", labelsize=7)
    ax.xaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    for i, p in enumerate(top_p[::-1]):
        ax.text(p + 3e-4, i, f"{p:.4f}", va="center", fontsize=7)

# ─── 5. Render function ───────────────────────────────────────────────────────
_out = widgets.Output()

def _render(puzzle_id: int, seq_pos: int) -> None:
    seq      = session.sequences[puzzle_id]
    tok      = int(seq[seq_pos])
    grid_str = build_grid_at_step(seq, seq_pos)
    logits_v = _logits(puzzle_id, seq_pos)

    with _out:
        _out.clear_output(wait=True)
        fig, (ax_board, ax_top10) = plt.subplots(
            1, 2, figsize=(13, 6.5),
            gridspec_kw={"width_ratios": [1.1, 1]},
        )
        fig.suptitle(
            f"Puzzle {puzzle_id}  │  seq_pos={seq_pos}"
            f"  │  token={tok}  │  {_decode(tok)}",
            fontsize=10,
        )
        _draw_board(ax_board, grid_str)
        ax_board.set_title(
            f"Board state + candidates  (filled={81 - grid_str.count('0')})",
            fontsize=9, pad=4,
        )
        _draw_top10(ax_top10, logits_v)
        plt.tight_layout()
        plt.show()

# ─── 6. Widgets ───────────────────────────────────────────────────────────────
_all_ids = sorted(set(int(p) for p in session.index.puzzle_idx))

_puzzle_dd = widgets.Dropdown(
    options=_all_ids,
    value=_all_ids[0],
    description="Puzzle ID:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px"),
)

_token_sel = widgets.Select(
    options=_token_options(_all_ids[0]),
    value=0,
    rows=20,
    description="Token:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="370px"),
)

_updating = False

def _on_puzzle_change(change):
    global _updating
    _updating = True
    opts = _token_options(change["new"])
    _token_sel.options = opts
    _token_sel.value   = opts[0][1]   # position 0
    _updating = False
    _render(change["new"], 0)

def _on_token_change(change):
    if _updating or change["new"] is None:
        return
    _render(_puzzle_dd.value, change["new"])

_puzzle_dd.observe(_on_puzzle_change, names="value")
_token_sel.observe(_on_token_change,  names="value")
_render(_all_ids[0], 0)

display(widgets.HBox([
    widgets.VBox([_puzzle_dd, _token_sel]),
    _out,
]))